# Docling PDF 파싱 성능 검증 (Parsing Evaluation / PoC)

이 노트북은 **PDF → Markdown 변환 테스트가 아니다.** Docling이 PDF의 문서 구조를 얼마나 정확히
복원하는지 측정하기 위한 **평가 노트북**이며, 실행 후 다음 세 질문에 답할 수 있어야 한다.

| # | 평가 영역 | 핵심 질문 |
|---|---|---|
| 1 | **Layout Analysis** | Text / Heading / Table / Picture / List / Header / Footer 영역을 올바른 문서 요소로 구분하는가? |
| 2 | **Text Hierarchy** | Heading → Subheading → Paragraph 계층과 문서 내 순서를 보존하는가? |
| 3 | **Table Structure** | 표의 Row / Column / Header / Cell / 병합 구조를 복원하는가? |

**대상 문서** — 사내 규정 `tiger_inc/pdf/*.pdf` 8종 + 법령 `tiger_inc/law/*.pdf` 3종, 총 11개 PDF를
한 번에 돌린다. 일부만 보려면 §2의 `TARGET_PDFS` 한 줄만 바꾼다.

**구성**

1. 환경 및 라이브러리 설정 → 2. PDF·경로 설정 → 3. Docling 문서 변환 → 4. Layout Analysis →
5. Text Hierarchy → 6. Table Structure → 7. 전체 결과 요약 → 8. Markdown 결과 저장

**실행 환경 주의 (이 머신 기준)**

- Docling은 base 파이썬이 아니라 conda env `docling`에 설치되어 있다
  (`C:\Users\young\miniconda3\envs\docling\python.exe` — docling 2.119.0 / Python 3.12).
  **반드시 해당 env의 커널로 실행**할 것.
- 실행 산출물은 전부 `docling_eval/output/` 아래에만 생성된다.
- 전체 11개 문서(222페이지) 변환은 CPU 기준 수 분~십수 분이 걸린다.

---
## 1. 환경 및 라이브러리 설정

`TORCHDYNAMO_DISABLE=1`을 **torch/docling import 이전에** 설정한다. 레이아웃 모델이 `torch.compile`을
쓰는데, MSVC(`cl.exe`)·triton이 없는 환경에서는 전 페이지가 `Compiler: cl is not found`로 실패해
변환 전체가 `status=failure`로 끝난다. import 뒤에 설정하면 효과가 없다.

In [1]:
import os

# ── torch / docling import 보다 반드시 먼저 (위 설명 참조) ──────────────────
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")

import json
import sys
import time
import traceback
from importlib.metadata import version as pkg_version
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import ContentLayer, DocItemLabel
from docling_core.types.doc.document import ListItem, SectionHeaderItem, TableItem

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 200)

print(f"python       : {sys.version.split()[0]}")
print(f"executable   : {sys.executable}")
for _pkg in ("docling", "docling-core", "pandas"):
    print(f"{_pkg:13}: {pkg_version(_pkg)}")

python       : 3.12.13
executable   : c:\Users\young\miniconda3\envs\docling\python.exe
docling      : 2.119.0
docling-core : 2.91.0
pandas       : 3.0.5


---
## 2. PDF 파일 및 경로 설정

- `BASE_DIR`는 노트북을 **레포 루트에서 열든 `docling_eval/` 안에서 열든** 항상 같은
  `.../docling_eval` 하나를 가리킨다(`docling_eval/docling_eval` 중복 방지).
- **대상 문서를 바꾸려면 `TARGET_PDFS` 한 줄만 수정**한다. `None`이면 `tiger_inc/pdf`와
  `tiger_inc/law`의 모든 PDF를 대상으로 한다.
- PDF를 하나도 못 찾으면 어디를 뒤졌는지 함께 담아 `FileNotFoundError`를 낸다.

In [2]:
def _resolve_base_dir() -> Path:
    """CWD가 레포 루트든 docling_eval 내부든 같은 docling_eval 을 가리키게 한다."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if cand.name == "docling_eval":
            return cand
        if (cand / "docling_eval").is_dir() or (cand / "tiger_inc").is_dir():
            return cand / "docling_eval"
    return cwd / "docling_eval"


BASE_DIR = _resolve_base_dir()
REPO_ROOT = BASE_DIR.parent

OUTPUT_DIR = BASE_DIR / "output"
LAYOUT_DIR = OUTPUT_DIR / "layout"
HIER_DIR = OUTPUT_DIR / "hierarchy"
TABLE_DIR = OUTPUT_DIR / "tables"
MD_DIR = OUTPUT_DIR / "markdown"
for _d in (OUTPUT_DIR, LAYOUT_DIR, HIER_DIR, TABLE_DIR, MD_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ▼▼▼ 검증 대상 — 바꿀 곳은 여기뿐 ▼▼▼
PDF_DIRS = [REPO_ROOT / "tiger_inc" / "pdf", REPO_ROOT / "tiger_inc" / "law"]
TARGET_PDFS = None      # None = 위 폴더의 전체 PDF. 일부만: ["법인카드_사용규정", "법인세법"]
# ▲▲▲

PDF_PATHS = []
for _dir in PDF_DIRS:
    if not _dir.is_dir():
        print(f"[warn] 폴더가 없습니다: {_dir}")
        continue
    for _p in sorted(_dir.glob("*.pdf")):
        if TARGET_PDFS is None or _p.stem in TARGET_PDFS:
            PDF_PATHS.append(_p)

if not PDF_PATHS:
    raise FileNotFoundError(
        "대상 PDF를 찾지 못했습니다.\n"
        + "".join(f"  탐색: {d}\n" for d in PDF_DIRS)
        + f"  TARGET_PDFS = {TARGET_PDFS}\n"
        "→ 위 셀의 PDF_DIRS / TARGET_PDFS 를 확인하세요."
    )

print(f"BASE_DIR   : {BASE_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"대상 PDF   : {len(PDF_PATHS)}개\n")

target_df = pd.DataFrame(
    [
        {
            "Document": p.stem,
            "Folder": p.parent.name,
            "Size (KB)": round(p.stat().st_size / 1024, 1),
            "Path": p.relative_to(REPO_ROOT).as_posix(),
        }
        for p in PDF_PATHS
    ]
)
display(target_df)

BASE_DIR   : D:\project\SKN29-FINAL-1TEAM\docling_eval
OUTPUT_DIR : D:\project\SKN29-FINAL-1TEAM\docling_eval\output
대상 PDF   : 11개



,Document,Folder,Size (KB),Path
0,법인카드_사용규정,pdf,612.5,tiger_inc/pdf/법인카드_사용규정.pdf
1,부서소개,pdf,298.7,tiger_inc/pdf/부서소개.pdf
2,업무추진비_사용규정,pdf,598.1,tiger_inc/pdf/업무추진비_사용규정.pdf
3,조직도,pdf,706.4,tiger_inc/pdf/조직도.pdf
4,조직설계_상세기획서,pdf,782.5,tiger_inc/pdf/조직설계_상세기획서.pdf
5,직급체계,pdf,517.6,tiger_inc/pdf/직급체계.pdf
6,출장비_사용규정,pdf,585.2,tiger_inc/pdf/출장비_사용규정.pdf
7,회식_운영규정,pdf,637.1,tiger_inc/pdf/회식_운영규정.pdf
8,법인세법,law,570.3,tiger_inc/law/법인세법.pdf
9,부가가치세법,law,269.5,tiger_inc/law/부가가치세법.pdf


---
## 3. Docling 문서 변환

파이프라인 옵션 세 가지가 이 평가의 전제다.

- `do_table_structure=True` + `do_cell_matching=True` — 표를 텍스트가 아니라 **셀 격자**로 복원한다(§6에서 검증).
- `heading_hierarchy_options.enabled=True` — **기본값이 `False`**이고, 그 상태에서는 모든 `section_header`가
  `level=1`로 평탄화되어 계층 검증(§5)이 무의미해진다. 켜면 제N장 → H1, 제N조 → H2 구조가 살아난다.
- `do_ocr=False` — 대상이 전부 텍스트 PDF다. 스캔본이라면 `True`로.

백엔드는 `pypdfium2` → 기본 `docling-parse` 순으로 시도한다. 한글 PDF·한글 경로에서 기본 백엔드가
`'utf-8' codec can't decode byte ...` / `Failed to load document with key` 로 실패하는 사례가 있어
폴백을 둔다. 어느 백엔드가 쓰였는지는 결과 표의 `Backend` 열에 남는다.

> 11개 문서 전체 변환은 CPU에서 수 분~십수 분이 걸린다. 문서 하나가 실패해도 나머지는 계속 진행하고,
> 실패 사유는 `[FAIL]` 줄과 결과 표의 `Note`에 남는다.

In [3]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.do_cell_matching = True
pipeline_options.do_ocr = False                              # 텍스트 PDF 기준
pipeline_options.heading_hierarchy_options.enabled = True    # ← 계층 검증(§5)의 핵심 스위치


def _build_converter(backend):
    fmt = PdfFormatOption(pipeline_options=pipeline_options)
    if backend is not None:
        fmt.backend = backend
    return DocumentConverter(format_options={InputFormat.PDF: fmt})


# 모델 로딩 비용을 아끼려 컨버터는 한 번만 만들어 전 문서에 재사용한다.
CONVERTERS = [
    ("pypdfium2", _build_converter(PyPdfiumDocumentBackend)),
    ("docling-parse (default)", _build_converter(None)),
]
OK_STATUS = (ConversionStatus.SUCCESS, ConversionStatus.PARTIAL_SUCCESS)

DOCS = {}          # 문서명 -> DoclingDocument
conv_rows = []

print("=" * 78)
print("Docling Document Conversion")
print("=" * 78)

for i, pdf in enumerate(PDF_PATHS, start=1):
    name = pdf.stem
    doc = backend_used = None
    status, n_errors, note = "FAILURE", 0, ""
    t0 = time.perf_counter()

    for label, converter in CONVERTERS:
        try:
            res = converter.convert(pdf)
        except Exception as exc:  # noqa: BLE001 — 백엔드별 예외를 모아 폴백한다
            note = f"{label}: {type(exc).__name__}: {exc}"[:160]
            continue
        if res.status in OK_STATUS:
            doc, backend_used = res.document, label
            status, n_errors = res.status.value.upper(), len(res.errors)
            break
        note = f"{label}: status={res.status.value}"

    elapsed = time.perf_counter() - t0
    row = {
        "Document": name, "Folder": pdf.parent.name, "Status": status,
        "Backend": backend_used or "-", "Pages": doc.num_pages() if doc else 0,
        "Texts": len(doc.texts) if doc else 0, "Tables": len(doc.tables) if doc else 0,
        "Pictures": len(doc.pictures) if doc else 0, "Groups": len(doc.groups) if doc else 0,
        "Errors": n_errors, "Sec": round(elapsed, 1), "Note": note,
    }
    conv_rows.append(row)

    if doc is None:
        print(f"[{i:2}/{len(PDF_PATHS)}] [FAIL] {name}  ← {note}")
    else:
        DOCS[name] = doc
        print(
            f"[{i:2}/{len(PDF_PATHS)}] {name:24.24s} pages={row['Pages']:3}  {status:15.15s}"
            f"  backend={backend_used:10.10s}  texts={row['Texts']:4}  tables={row['Tables']:3}"
            f"  pics={row['Pictures']:3}  {elapsed:6.1f}s"
        )

conversion_df = pd.DataFrame(conv_rows)
print(f"\n변환 성공 {len(DOCS)} / {len(PDF_PATHS)} 건")
if not DOCS:
    raise RuntimeError("모든 문서 변환이 실패했습니다. 위 [FAIL] 사유와 Note 열을 확인하세요.")
display(conversion_df)

Docling Document Conversion


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[ 1/11] 법인카드_사용규정                pages=  7  SUCCESS          backend=pypdfium2   texts= 118  tables=  2  pics=  0     3.8s
[ 2/11] 부서소개                     pages=  5  SUCCESS          backend=pypdfium2   texts=  23  tables=  9  pics=  0     1.2s
[ 3/11] 업무추진비_사용규정               pages=  6  SUCCESS          backend=pypdfium2   texts=  99  tables=  2  pics=  0     0.5s
[ 4/11] 조직도                      pages=  6  SUCCESS          backend=pypdfium2   texts=  86  tables=  4  pics=  1     0.6s
[ 5/11] 조직설계_상세기획서               pages= 12  SUCCESS          backend=pypdfium2   texts= 137  tables= 12  pics=  1     2.0s
[ 6/11] 직급체계                     pages=  4  SUCCESS          backend=pypdfium2   texts=  28  tables=  3  pics=  0     0.6s
[ 7/11] 출장비_사용규정                 pages=  6  SUCCESS          backend=pypdfium2   texts=  79  tables=  4  pics=  0     0.6s
[ 8/11] 회식_운영규정                  pages= 10  SUCCESS          backend=pypdfium2   texts=  94  tables= 11  pics=  0     2.0s
[ 9/11] 법인세법    

,Document,Folder,Status,Backend,Pages,Texts,Tables,Pictures,Groups,Errors,Sec,Note
0,법인카드_사용규정,pdf,SUCCESS,pypdfium2,7,118,2,0,16,0,3.8,
1,부서소개,pdf,SUCCESS,pypdfium2,5,23,9,0,0,0,1.2,
2,업무추진비_사용규정,pdf,SUCCESS,pypdfium2,6,99,2,0,10,0,0.5,
3,조직도,pdf,SUCCESS,pypdfium2,6,86,4,1,4,0,0.6,
4,조직설계_상세기획서,pdf,SUCCESS,pypdfium2,12,137,12,1,13,0,2.0,
5,직급체계,pdf,SUCCESS,pypdfium2,4,28,3,0,5,0,0.6,
6,출장비_사용규정,pdf,SUCCESS,pypdfium2,6,79,4,0,10,0,0.6,
7,회식_운영규정,pdf,SUCCESS,pypdfium2,10,94,11,0,5,0,2.0,
8,법인세법,law,SUCCESS,pypdfium2,97,2093,4,3,258,0,7.6,
9,부가가치세법,law,SUCCESS,pypdfium2,35,763,3,1,87,0,1.9,


---
## 4. Layout Analysis 검증

> **Docling이 PDF 페이지의 서로 다른 영역을 올바른 문서 요소로 구분하는가?**

`doc.iterate_items()`로 문서 트리를 리딩오더대로 순회하며 요소마다
**Page / Element Type / Bounding Box / Text / Order**를 뽑는다.

- 머리말·꼬리말(`page_header`/`page_footer`)은 docling이 `FURNITURE` 콘텐츠 레이어에 넣고
  기본 순회에서 **제외**한다. Header/Footer 인식률도 평가 대상이므로 `BODY ∪ FURNITURE`로 순회한다.
- docling 원본 라벨(30종)은 CSV의 `Docling Label`에 그대로 남기고, 화면 집계는
  Title/Heading/Text/List/Table/Picture/Caption/Header/Footer/… 로 묶어 본다.

결과는 `output/layout/layout_result.csv`(요소 단위 전수) 및 집계 CSV 2종으로 저장한다.

In [4]:
# docling 원본 라벨 → 화면 집계용 요소 타입
ELEMENT_GROUP = {
    "title": "Title", "section_header": "Heading",
    "text": "Text", "paragraph": "Text",
    "list_item": "List",
    "table": "Table", "document_index": "Table",
    "picture": "Picture", "chart": "Picture",
    "caption": "Caption", "footnote": "Footnote",
    "page_header": "Header", "page_footer": "Footer",
    "formula": "Formula", "code": "Code", "reference": "Reference",
    "form": "Form", "key_value_region": "Form",
}
# 머리말/꼬리말을 세려면 FURNITURE 를 포함해야 한다(기본 순회는 BODY 만).
INCLUDED_LAYERS = {ContentLayer.BODY, ContentLayer.FURNITURE}


def _element_text(item, doc) -> str:
    if isinstance(item, TableItem):
        return item.caption_text(doc) or f"<table {item.data.num_rows}x{item.data.num_cols}>"
    text = getattr(item, "text", None)
    if text:
        return " ".join(text.split())
    label = getattr(getattr(item, "label", None), "value", "item")
    return f"<{label}>"


def _heading_level(item, label: str) -> str:
    if isinstance(item, SectionHeaderItem):
        return f"H{item.level}"
    return "H1" if label == "title" else ""


def collect_elements(name: str, doc) -> list[dict]:
    rows = []
    items = doc.iterate_items(with_groups=False, included_content_layers=INCLUDED_LAYERS)
    for order, (item, depth) in enumerate(items, start=1):
        label = getattr(getattr(item, "label", None), "value", None)
        if label is None:       # 라벨 없는 컨테이너 노드는 건너뛴다
            continue
        prov = list(getattr(item, "prov", None) or [])
        bbox = prov[0].bbox if prov else None
        text = _element_text(item, doc)
        rows.append({
            "Document": name,
            "Order": order,
            "Page": prov[0].page_no if prov else None,
            "Element Type": ELEMENT_GROUP.get(label, "Other"),
            "Docling Label": label,
            "Level": _heading_level(item, label),
            "Depth": depth,
            "Marker": (getattr(item, "marker", "") or "") if isinstance(item, ListItem) else "",
            "BBox (l,t,r,b)": (
                f"{bbox.l:.1f},{bbox.t:.1f},{bbox.r:.1f},{bbox.b:.1f}" if bbox else ""
            ),
            "Chars": len(text),
            "Text": text,
        })
    return rows


layout_rows = []
for _name, _doc in DOCS.items():
    try:
        layout_rows.extend(collect_elements(_name, _doc))
    except Exception:  # noqa: BLE001 — 한 문서가 깨져도 나머지는 집계한다
        print(f"[error] Layout 수집 실패: {_name}")
        traceback.print_exc()

layout_df = pd.DataFrame(layout_rows)
layout_df.to_csv(LAYOUT_DIR / "layout_result.csv", index=False, encoding="utf-8-sig")
print(f"요소 {len(layout_df):,}개 수집 → {(LAYOUT_DIR / 'layout_result.csv').relative_to(BASE_DIR)}")

_unknown = sorted(set(layout_df.loc[layout_df["Element Type"] == "Other", "Docling Label"]))
if _unknown:
    print(f"[note] 매핑되지 않은 docling 라벨(Other 로 집계): {_unknown}")

display(layout_df.head(15))

요소 4,388개 수집 → output\layout\layout_result.csv


,Document,Order,Page,Element Type,Docling Label,Level,Depth,Marker,"BBox (l,t,r,b)",Chars,Text
0,법인카드_사용규정,1,1,Heading,section_header,H1,1,,"206.1,594.1,387.6,580.0",34,타 이 거 주 식 회 사 ( T i ge r I n c . )
1,법인카드_사용규정,2,1,Heading,section_header,H1,1,,"183.0,511.9,409.3,486.4",10,법인카드 사용 규정
2,법인카드_사용규정,3,1,Table,table,,1,,"205.0,378.9,389.8,292.6",11,<table 3x2>
3,법인카드_사용규정,4,1,Text,text,,1,,"113.3,272.4,481.4,251.2",100,"개정이력 v1.0 제정 (조문 정합성 검토 반영) / v1.1 개정: 제2조 관리자 정의 정비, 제10조 사전승인 예외..."
4,법인카드_사용규정,5,1,Text,text,,1,,"91.5,95.7,503.8,63.4",212,"※ 본 규정은 데모/교육 목적의 가상 기업(타이거 주식회사, Tiger Inc.) 시나리오용으로 작성되었으며, 회사는 ..."
5,법인카드_사용규정,6,2,Header,page_header,,1,,"265.3,811.5,329.8,803.8",10,법인카드 사용 규정
6,법인카드_사용규정,7,2,Heading,section_header,H2,1,,"62.6,666.8,120.8,654.9",8,제1조 (목적)
7,법인카드_사용규정,8,2,Text,text,,1,,"62.6,641.1,532.8,597.0",152,"이 규정은 타이거 주식회사(이하 ""회사"")가 임직원에게 발급하는 법인카드의 발급, 관리, 사용 및 정산에 관 한 사항을..."
8,법인카드_사용규정,9,2,Heading,section_header,H2,1,,"62.6,572.0,160.4,560.1",16,제2조 (정의) 개정 v1.1
9,법인카드_사용규정,10,2,Text,text,,1,,"63.4,546.4,279.5,536.3",27,이 규정에서 사용하는 용어의 정의는 다음과 같다.


### 4-1. 요소 타입 집계 — 문서별 / 전체 / 페이지별

In [5]:
# (a) 문서 × 요소 타입
layout_by_document = (
    pd.crosstab(layout_df["Document"], layout_df["Element Type"], margins=True, margins_name="ALL")
    .sort_index()
)
layout_by_document.to_csv(LAYOUT_DIR / "layout_by_document.csv", encoding="utf-8-sig")
print("[a] 문서별 요소 타입 집계")
display(layout_by_document)

# (b) 전체 요소 타입
layout_totals = (
    layout_df["Element Type"].value_counts().rename_axis("Element Type").reset_index(name="Count")
)
layout_totals["Ratio"] = (layout_totals["Count"] / len(layout_df)).round(3)
print("[b] 전체 문서 요소 타입 합계")
display(layout_totals)

# (c) 페이지별 (전 문서 전수 저장 + 한 문서 미리보기)
layout_by_page = (
    layout_df.groupby(["Document", "Page", "Element Type"], dropna=False)
    .size().reset_index(name="Count")
)
layout_by_page.to_csv(LAYOUT_DIR / "layout_by_page.csv", index=False, encoding="utf-8-sig")

PREVIEW_DOC = next(iter(DOCS))          # 미리보기 문서 — 바꿔가며 확인 가능
_prev = layout_df[layout_df["Document"] == PREVIEW_DOC]
print(f"[c] 페이지별 요소 타입 — 미리보기: {PREVIEW_DOC} (전 문서는 layout_by_page.csv)")
display(pd.crosstab(_prev["Page"], _prev["Element Type"], margins=True, margins_name="ALL"))

[a] 문서별 요소 타입 집계


Element Type,Caption,Code,Footer,Formula,Header,Heading,List,Picture,Table,Text,ALL
Document,,,,,,,,,,,
ALL,2,14,431,3,214,493,2432,7,54,738,4388
법인세법,0,6,194,3,97,188,1216,3,4,387,2098
법인카드_사용규정,1,0,12,0,6,28,60,0,2,11,120
부가가치세법,0,3,70,0,35,57,493,1,3,105,767
부서소개,0,0,9,0,4,9,0,0,9,1,32
업무추진비_사용규정,0,0,10,0,5,25,41,0,2,18,101
여신전문금융업법,0,5,68,0,34,103,530,1,0,167,908
조직도,1,0,10,0,5,9,9,1,4,3,42
조직설계_상세기획서,0,0,22,0,11,23,25,1,12,7,101


[b] 전체 문서 요소 타입 합계


,Element Type,Count,Ratio
0,List,2432,0.554
1,Text,738,0.168
2,Heading,493,0.112
3,Footer,431,0.098
4,Header,214,0.049
5,Table,54,0.012
6,Code,14,0.003
7,Picture,7,0.002
8,Formula,3,0.001
9,Caption,2,0.000


[c] 페이지별 요소 타입 — 미리보기: 법인카드_사용규정 (전 문서는 layout_by_page.csv)


Element Type,Caption,Footer,Header,Heading,List,Table,Text,ALL
Page,,,,,,,,
1,0,0,0,2,0,1,2,5
2,0,2,1,4,6,0,3,16
3,0,2,1,7,11,0,4,25
4,0,2,1,4,16,0,0,23
5,0,2,1,5,13,0,1,22
6,0,2,1,6,9,0,1,19
7,1,2,1,0,5,1,0,10
ALL,1,12,6,28,60,2,11,120


---
## 5. Text Hierarchy 검증

> **Heading → Subheading → Paragraph 계층과 문서 내 순서를 보존하는가?**

`SectionHeaderItem.level`(§3에서 `heading_hierarchy_options`를 켜서 살아난 값)을 스택으로 삼아
본문·목록·표를 직전 헤딩 아래에 붙여 트리를 그린다. 화면에는 헤딩당 본문 3개까지만 접어서
보여주고, **접지 않은 전체 트리는 `output/hierarchy/<문서>_tree.txt`** 에 저장한다.

계층 품질은 눈으로만 보지 않고 세 지표로 함께 잰다.

| 지표 | 의미 | 나쁠 때 |
|---|---|---|
| `Level Jumps` | H1 → H3 처럼 레벨을 건너뛴 횟수 | 중간 헤딩을 본문으로 오인 |
| `Orphan Body` | 첫 헤딩보다 앞에 나온 본문 요소 수 | 표지·머리말이 본문에 섞임 (일부는 정상) |
| `Flat?` | 헤딩이 전부 H1인가 | 계층이 평탄화됨 (= 사실상 계층 복원 실패) |

In [6]:
HEADING_TYPES = {"Title", "Heading"}
BODY_TYPES = {"Text", "List", "Table", "Picture", "Caption", "Formula", "Code", "Footnote"}
TREE_TAG = {
    "Text": "TEXT", "List": "LIST", "Table": "TABLE", "Picture": "PIC",
    "Caption": "CAP", "Formula": "FORM", "Code": "CODE", "Footnote": "NOTE",
}

hierarchy_df = (
    layout_df[layout_df["Element Type"].isin(HEADING_TYPES | BODY_TYPES)]
    [["Document", "Order", "Page", "Element Type", "Level", "Marker", "Text"]]
    .rename(columns={"Element Type": "Type"})
    .reset_index(drop=True)
)
hierarchy_df.to_csv(HIER_DIR / "hierarchy_result.csv", index=False, encoding="utf-8-sig")


def render_tree(doc_name: str, max_body_per_heading: int = 3, text_width: int = 62) -> list[str]:
    """헤딩 레벨을 스택 삼아 문서 계층을 텍스트 트리로 그린다."""
    lines, cur_depth, shown = [], 0, 0
    for r in hierarchy_df[hierarchy_df["Document"] == doc_name].to_dict("records"):
        text = r["Text"]
        if len(text) > text_width:
            text = text[:text_width] + "…"
        if r["Type"] in HEADING_TYPES:
            level = r["Level"]
            depth = max(int(level[1:]) - 1, 0) if level.startswith("H") else 0
            cur_depth, shown = depth + 1, 0
            lines.append(f"{'│   ' * depth}├─ [{level or 'H?'}] {text}   (p.{r['Page']})")
        else:
            shown += 1
            if shown > max_body_per_heading:
                if shown == max_body_per_heading + 1:
                    lines.append(f"{'│   ' * cur_depth}└─ … (이하 생략)")
                continue
            marker = f"{r['Marker']} " if r["Marker"] else ""
            lines.append(f"{'│   ' * cur_depth}├─ [{TREE_TAG.get(r['Type'], r['Type'])}] {marker}{text}")
    return lines


def hierarchy_metrics(doc_name: str) -> dict:
    rows = hierarchy_df[hierarchy_df["Document"] == doc_name].to_dict("records")
    levels, jumps, orphan, seen_heading = [], 0, 0, False
    prev = 0
    for r in rows:
        if r["Type"] in HEADING_TYPES and r["Level"].startswith("H"):
            lvl = int(r["Level"][1:])
            levels.append(lvl)
            if prev and lvl > prev + 1:
                jumps += 1
            prev, seen_heading = lvl, True
        elif not seen_heading:
            orphan += 1
    return {
        "Document": doc_name,
        "Headings": len(levels),
        "Max Level": max(levels) if levels else 0,
        "Level Jumps": jumps,
        "Orphan Body": orphan,
        "Flat?": "YES (계층 없음)" if levels and max(levels) == 1 else "no",
    }


# 헤딩 레벨 분포 + 계층 품질 지표
_head = hierarchy_df[hierarchy_df["Type"].isin(HEADING_TYPES) & hierarchy_df["Level"].ne("")]
heading_level_df = pd.crosstab(_head["Document"], _head["Level"], margins=True, margins_name="ALL")
hierarchy_metrics_df = pd.DataFrame([hierarchy_metrics(n) for n in DOCS])

print("[a] 문서별 Heading 레벨 분포")
display(heading_level_df)
print("[b] 계층 품질 지표")
display(hierarchy_metrics_df)

# 전체 트리는 파일로, 화면에는 미리보기 문서만
for _name in DOCS:
    (HIER_DIR / f"{_name}_tree.txt").write_text(
        "\n".join(render_tree(_name, max_body_per_heading=10**9)), encoding="utf-8"
    )
print(f"\n전체 계층 트리 {len(DOCS)}건 저장 → {HIER_DIR.relative_to(BASE_DIR)}/<문서>_tree.txt")

[a] 문서별 Heading 레벨 분포


Level,H1,H2,ALL
Document,,,
법인세법,35,153,188
법인카드_사용규정,9,19,28
부가가치세법,10,47,57
부서소개,9,0,9
업무추진비_사용규정,9,16,25
여신전문금융업법,11,92,103
조직도,9,0,9
조직설계_상세기획서,21,2,23
직급체계,6,0,6


[b] 계층 품질 지표


,Document,Headings,Max Level,Level Jumps,Orphan Body,Flat?
0,법인카드_사용규정,28,2,0,0,no
1,부서소개,9,1,0,0,YES (계층 없음)
2,업무추진비_사용규정,25,2,0,0,no
3,조직도,9,1,0,0,YES (계층 없음)
4,조직설계_상세기획서,23,2,0,1,no
5,직급체계,6,1,0,0,YES (계층 없음)
6,출장비_사용규정,24,2,0,0,no
7,회식_운영규정,21,2,0,0,no
8,법인세법,188,2,0,0,no
9,부가가치세법,57,2,0,0,no



전체 계층 트리 11건 저장 → output\hierarchy/<문서>_tree.txt


In [7]:
# 미리보기 트리 — PREVIEW_DOC 를 바꾸면 다른 문서를 볼 수 있다
TREE_PREVIEW_LINES = 45

print("=" * 78)
print(f"DOCUMENT HIERARCHY — {PREVIEW_DOC}")
print("=" * 78)
_lines = render_tree(PREVIEW_DOC)
for _line in _lines[:TREE_PREVIEW_LINES]:
    print(_line)
if len(_lines) > TREE_PREVIEW_LINES:
    print(f"... ({len(_lines) - TREE_PREVIEW_LINES}줄 더 — 전체는 hierarchy/{PREVIEW_DOC}_tree.txt)")

print("\n[요소 순서 표 — 앞 20행]")
display(hierarchy_df[hierarchy_df["Document"] == PREVIEW_DOC].head(20))

DOCUMENT HIERARCHY — 법인카드_사용규정
├─ [H1] 타 이 거 주 식 회 사 ( T i ge r I n c . )   (p.1)
├─ [H1] 법인카드 사용 규정   (p.1)
│   ├─ [TABLE] <table 3x2>
│   ├─ [TEXT] 개정이력 v1.0 제정 (조문 정합성 검토 반영) / v1.1 개정: 제2조 관리자 정의 정비, 제10조 사전승…
│   ├─ [TEXT] ※ 본 규정은 데모/교육 목적의 가상 기업(타이거 주식회사, Tiger Inc.) 시나리오용으로 작성되었으며, …
│   ├─ [H2] 제1조 (목적)   (p.2)
│   │   ├─ [TEXT] 이 규정은 타이거 주식회사(이하 "회사")가 임직원에게 발급하는 법인카드의 발급, 관리, 사용 및 정산에 관 한…
│   ├─ [H2] 제2조 (정의) 개정 v1.1   (p.2)
│   │   ├─ [TEXT] 이 규정에서 사용하는 용어의 정의는 다음과 같다.
│   │   ├─ [LIST] 1. "법인카드"란 회사 명의로 발급되어 임직원이 업무 목적의 지출에 사용하는 신용카드·체크카드를 말한 다.
│   │   ├─ [LIST] 2. "사용자"란 법인카드를 발급받아 사용하는 임직원을 말한다.
│   │   └─ … (이하 생략)
│   ├─ [H2] 제3조 (적용범위)   (p.2)
│   │   ├─ [TEXT] 이 규정은 회사로부터 법인카드를 발급받은 모든 임직원에게 적용한다.
├─ [H1] 제1장 총칙   (p.2)
│   ├─ [H2] 제4조 (발급 대상)   (p.3)
│   │   ├─ [LIST] 팀장 이상 직책(팀장·부서장·본부장·대표이사)을 보임한 임직원에게는 원칙적으로 개인 법인카드를 발 급한다. 1.
│   │   ├─ [LIST] 직책이 없는 임직원(비직책자)은 부서 공용카드를 사용하거나, 업무상 필요가 인정되는 경우 관리자 승 2.
│   │   ├─ [LIST] 인을 받아 개인 카드를 발급받을 수 있다.
│   ├─ [H2] 제5조 (발

,Document,Order,Page,Type,Level,Marker,Text
0,법인카드_사용규정,1,1,Heading,H1,,타 이 거 주 식 회 사 ( T i ge r I n c . )
1,법인카드_사용규정,2,1,Heading,H1,,법인카드 사용 규정
2,법인카드_사용규정,3,1,Table,,,<table 3x2>
3,법인카드_사용규정,4,1,Text,,,"개정이력 v1.0 제정 (조문 정합성 검토 반영) / v1.1 개정: 제2조 관리자 정의 정비, 제10조 사전승인 예외..."
4,법인카드_사용규정,5,1,Text,,,"※ 본 규정은 데모/교육 목적의 가상 기업(타이거 주식회사, Tiger Inc.) 시나리오용으로 작성되었으며, 회사는 ..."
5,법인카드_사용규정,7,2,Heading,H2,,제1조 (목적)
6,법인카드_사용규정,8,2,Text,,,"이 규정은 타이거 주식회사(이하 ""회사"")가 임직원에게 발급하는 법인카드의 발급, 관리, 사용 및 정산에 관 한 사항을..."
7,법인카드_사용규정,9,2,Heading,H2,,제2조 (정의) 개정 v1.1
8,법인카드_사용규정,10,2,Text,,,이 규정에서 사용하는 용어의 정의는 다음과 같다.
9,법인카드_사용규정,11,2,List,,1.,"""법인카드""란 회사 명의로 발급되어 임직원이 업무 목적의 지출에 사용하는 신용카드·체크카드를 말한 다."


---
## 6. Table Structure 검증

> **원본 PDF의 표가 텍스트 덩어리가 아니라 행/열/셀 구조로 얼마나 정확하게 복원되는가?**

`TableItem.data`(= `TableData`)의 셀 목록을 직접 뜯어 다음을 센다.

- `num_rows` / `num_cols` / 셀 수
- `column_header=True` 셀 → **헤더 검출 여부**, 헤더가 걸친 **행 수 → 다중 헤더**
- `row_header=True` 셀 → 좌측 헤더(레이블 열)
- `row_span>1 or col_span>1` → **병합 셀**
- 빈 셀 수와 채움률(`Fill Rate`) — 셀 매칭이 실패하면 여기서 먼저 티가 난다

표마다 `output/tables/<문서>/table_NN.csv`(DataFrame)와 `table_NN.json`(셀 단위 구조 + Markdown)을
남긴다. 화면에는 문서당 `TABLE_DETAIL_LIMIT`개만 상세 출력한다(저장은 전부).

> **`Rows=0, Cols=0`인 표를 특히 눈여겨볼 것.** docling이 "여기 표가 있다"는 영역은 잡았는데 셀 격자
> 복원에는 실패한 경우다. 표 개수만 세면 성공처럼 보이지만 실제로는 내용이 통째로 비어 있다.
> `Note` 열에 `표 영역만 검출 · 셀 구조 복원 실패 (0x0)`로 표시된다.

In [8]:
TABLE_DETAIL_LIMIT = 2      # 문서당 화면에 상세 출력할 표 수 (파일 저장은 전부)
TABLE_MD_LINES = 12         # 화면에 보여줄 Markdown 표의 최대 줄 수


def table_structure(name: str, doc, idx: int, table) -> dict:
    data = table.data
    cells = list(data.table_cells or [])
    header_cells = [c for c in cells if c.column_header]
    row_header_cells = [c for c in cells if c.row_header]
    merged = [c for c in cells if (c.row_span or 1) > 1 or (c.col_span or 1) > 1]
    empty = [c for c in cells if not (c.text or "").strip()]
    header_rows = sorted({c.start_row_offset_idx for c in header_cells})
    prov = list(table.prov or [])
    return {
        "Document": name, "Table": idx,
        "Page": prov[0].page_no if prov else None,
        "Rows": data.num_rows, "Cols": data.num_cols, "Cells": len(cells),
        "Header": "Detected" if header_cells else "Not detected",
        "Header Rows": len(header_rows),
        "Multi Header": "Y" if len(header_rows) > 1 else "N",
        "Row Header": "Y" if row_header_cells else "N",
        "Merged Cells": len(merged),
        "Empty Cells": len(empty),
        "Fill Rate": round(1 - len(empty) / len(cells), 3) if cells else 0.0,
        "Caption": (table.caption_text(doc) or "")[:60],
        # 표 영역은 잡았는데 셀이 0개 = 구조 복원 실패. 그냥 두면 Rows/Cols 0 이 조용히 묻힌다.
        "Note": "" if cells else "표 영역만 검출 · 셀 구조 복원 실패 (0x0)",
    }


table_rows, table_frames = [], {}

for name, doc in DOCS.items():
    if not doc.tables:
        continue
    out_dir = TABLE_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)

    for idx, table in enumerate(doc.tables, start=1):
        info = table_structure(name, doc, idx, table)
        stem = out_dir / f"table_{idx:02d}"

        try:
            df = table.export_to_dataframe(doc)
            df.to_csv(stem.with_suffix(".csv"), index=False, encoding="utf-8-sig")
            table_frames[(name, idx)] = df
        except Exception as exc:  # noqa: BLE001 — 표 하나가 깨져도 나머지는 진행
            info["Note"] = f"DataFrame 변환 실패: {type(exc).__name__}: {exc}"[:120]

        try:
            markdown_table = table.export_to_markdown(doc)
        except Exception as exc:  # noqa: BLE001
            markdown_table = ""
            info["Note"] = (info["Note"] + f" | Markdown 실패: {type(exc).__name__}").strip(" |")

        payload = dict(info)
        payload["cells"] = [
            {
                "row": c.start_row_offset_idx, "col": c.start_col_offset_idx,
                "row_span": c.row_span, "col_span": c.col_span,
                "column_header": c.column_header, "row_header": c.row_header,
                "text": c.text,
            }
            for c in (table.data.table_cells or [])
        ]
        payload["markdown"] = markdown_table
        stem.with_suffix(".json").write_text(
            json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
        )

        table_rows.append(info)

tables_df = pd.DataFrame(table_rows)
if tables_df.empty:
    print("[note] 어떤 문서에서도 표가 검출되지 않았습니다.")
else:
    tables_df.to_csv(TABLE_DIR / "table_summary.csv", index=False, encoding="utf-8-sig")
    print(f"표 {len(tables_df)}개 → {TABLE_DIR.relative_to(BASE_DIR)}/<문서>/table_NN.[csv|json]")
    display(tables_df)

표 54개 → output\tables/<문서>/table_NN.[csv|json]


,Document,Table,Page,Rows,Cols,Cells,Header,Header Rows,Multi Header,Row Header,Merged Cells,Empty Cells,Fill Rate,Caption,Note
0,법인카드_사용규정,1,1,3,2,6,Not detected,0,N,N,0,0,1.0,,
1,법인카드_사용규정,2,7,6,4,24,Detected,1,N,N,0,0,1.0,별표 1. 직책별 법인카드 사용 한도 개정 v1.1,
2,부서소개,1,2,6,5,30,Detected,1,N,N,0,0,1.0,,
3,부서소개,2,2,3,5,15,Detected,1,N,N,0,0,1.0,,
4,부서소개,3,3,3,5,15,Detected,1,N,N,0,0,1.0,,
5,부서소개,4,3,4,5,20,Detected,1,N,N,0,0,1.0,,
6,부서소개,5,3,3,5,15,Detected,1,N,N,0,0,1.0,,
7,부서소개,6,4,2,5,10,Detected,1,N,N,0,0,1.0,,
8,부서소개,7,4,2,5,10,Detected,1,N,N,0,0,1.0,,
9,부서소개,8,4,7,3,19,Detected,1,N,N,2,0,1.0,,


In [9]:
# 표 상세 — 문서당 TABLE_DETAIL_LIMIT 개
for name, doc in DOCS.items():
    for idx, table in enumerate(doc.tables[:TABLE_DETAIL_LIMIT], start=1):
        info = table_structure(name, doc, idx, table)
        print("=" * 78)
        print(f"TABLE {idx} — {name}")
        print("=" * 78)
        print(f"Page        : {info['Page']}")
        print(f"Rows        : {info['Rows']}")
        print(f"Columns     : {info['Cols']}")
        print(f"Header      : {info['Header']} (header rows={info['Header Rows']}, "
              f"multi={info['Multi Header']}, row-header={info['Row Header']})")
        print(f"Merged Cell : {'Detected (%d)' % info['Merged Cells'] if info['Merged Cells'] else 'None'}")
        print(f"Empty Cell  : {info['Empty Cells']}  (fill rate {info['Fill Rate']:.1%})")
        if info["Caption"]:
            print(f"Caption     : {info['Caption']}")
        if info["Note"]:
            print(f"Note        : {info['Note']}")

        df = table_frames.get((name, idx))
        if df is not None:
            display(df)

        md_lines = (table.export_to_markdown(doc) or "").splitlines()
        if md_lines:
            shown = md_lines[:TABLE_MD_LINES]
            if len(md_lines) > TABLE_MD_LINES:
                shown.append(f"| … ({len(md_lines) - TABLE_MD_LINES}줄 생략) |")
            display(Markdown("\n".join(shown)))
        print()

TABLE 1 — 법인카드_사용규정
Page        : 1
Rows        : 3
Columns     : 2
Header      : Not detected (header rows=0, multi=N, row-header=N)
Merged Cell : None
Empty Cell  : 0  (fill rate 100.0%)


,0,1
0,제정일,2026. 7. 20.
1,시행일,2026. 8. 1.
2,소관부서,경영지원본부


| 제정일    | 2026. 7. 20.   |
|-----------|----------------|
| 시행일    | 2026. 8. 1.    |
| 소관부서  | 경영지원본부   |


TABLE 2 — 법인카드_사용규정
Page        : 7
Rows        : 6
Columns     : 4
Header      : Detected (header rows=1, multi=N, row-header=N)
Merged Cell : None
Empty Cell  : 0  (fill rate 100.0%)
Caption     : 별표 1. 직책별 법인카드 사용 한도 개정 v1.1


,직책,1일 한도,월 한도,건당 사전승인 기준
0,대표이사,300만원,"1,000만원",100만원 초과
1,본부장,250만원,800만원,80만원 초과
2,부서장,150만원,500만원,60만원 초과
3,팀장,100만원,400만원,50만원 초과
4,비직책자(공용카드),50만원,200만원,30만원 초과


...

""



TABLE 1 — 부가가치세법
Page        : 5
Rows        : 0
Columns     : 0
Header      : Not detected (header rows=0, multi=N, row-header=N)
Merged Cell : None
Empty Cell  : 0  (fill rate 0.0%)
Note        : 표 영역만 검출 · 셀 구조 복원 실패 (0x0)


""



TABLE 2 — 부가가치세법
Page        : 19
Rows        : 0
Columns     : 0
Header      : Not detected (header rows=0, multi=N, row-header=N)
Merged Cell : None
Empty Cell  : 0  (fill rate 0.0%)
Note        : 표 영역만 검출 · 셀 구조 복원 실패 (0x0)


""


---
## 7. 전체 결과 요약

세 평가 영역(Layout / Hierarchy / Table)을 한 화면에서 확인한다.
문서별 수치는 `summary_df`(→ `output/summary.csv`)에 있다.

In [10]:
def _count(name: str, etype: str) -> int:
    return int(((layout_df["Document"] == name) & (layout_df["Element Type"] == etype)).sum())


summary_rows = []
for name in DOCS:
    conv = conversion_df.loc[conversion_df["Document"] == name].iloc[0]
    metrics = hierarchy_metrics(name)
    doc_tables = tables_df[tables_df["Document"] == name] if not tables_df.empty else pd.DataFrame()
    summary_rows.append({
        "Document": name,
        "Folder": conv["Folder"],
        "Pages": int(conv["Pages"]),
        "Status": conv["Status"],
        "Sec": conv["Sec"],
        "Elements": int((layout_df["Document"] == name).sum()),
        "Title": _count(name, "Title"),
        "Heading": _count(name, "Heading"),
        "Text": _count(name, "Text"),
        "List": _count(name, "List"),
        "Table": _count(name, "Table"),
        "Picture": _count(name, "Picture"),
        "Header/Footer": _count(name, "Header") + _count(name, "Footer"),
        "Max Level": metrics["Max Level"],
        "Level Jumps": metrics["Level Jumps"],
        "Flat?": metrics["Flat?"],
        "Tables Converted": int(len(doc_tables)),
        "Tables 0x0": int((doc_tables["Cells"] == 0).sum()) if len(doc_tables) else 0,
        "Table Rows": int(doc_tables["Rows"].sum()) if len(doc_tables) else 0,
        "Merged Cells": int(doc_tables["Merged Cells"].sum()) if len(doc_tables) else 0,
        "Header Detected": int((doc_tables["Header"] == "Detected").sum()) if len(doc_tables) else 0,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "summary.csv", index=False, encoding="utf-8-sig")

_levels = (
    hierarchy_df[hierarchy_df["Type"].isin(HEADING_TYPES) & hierarchy_df["Level"].ne("")]["Level"]
    .value_counts().sort_index()
)

print("=" * 78)
print("DOCLING PARSING TEST SUMMARY")
print("=" * 78)
print(f"Documents          : {len(DOCS)} / {len(PDF_PATHS)} 변환 성공")
print(f"Pages              : {int(summary_df['Pages'].sum())}")
print(f"Elapsed            : {summary_df['Sec'].sum():.1f}s")

print("\n[1] Layout Analysis")
print("-" * 40)
for _t, _c in layout_df["Element Type"].value_counts().items():
    print(f"{_t:<18} : {_c}")

print("\n[2] Text Hierarchy")
print("-" * 40)
print(f"{'Heading detected':<18} : {int(_levels.sum())}")
for _lvl, _c in _levels.items():
    print(f"{_lvl:<18} : {_c}")
print(f"{'Level jumps':<18} : {int(summary_df['Level Jumps'].sum())}")
print(f"{'Flat documents':<18} : {int((summary_df['Flat?'] != 'no').sum())} / {len(summary_df)}")

print("\n[3] Table Structure")
print("-" * 40)
if tables_df.empty:
    print("Tables detected    : 0")
else:
    _structured = tables_df[tables_df["Cells"] > 0]
    print(f"{'Tables detected':<18} : {len(tables_df)}")
    print(f"{'Tables converted':<18} : {len(table_frames)}")
    print(f"{'Structure OK':<18} : {len(_structured)} / {len(tables_df)}"
          f"   (0x0 = 영역만 검출: {len(tables_df) - len(_structured)})")
    print(f"{'Rows reconstructed':<18} : {int(tables_df['Rows'].sum())}")
    print(f"{'Columns (max)':<18} : {int(tables_df['Cols'].max())}")
    print(f"{'Header detected':<18} : {int((tables_df['Header'] == 'Detected').sum())} / {len(tables_df)}")
    print(f"{'Multi header':<18} : {int((tables_df['Multi Header'] == 'Y').sum())}")
    print(f"{'Merged cells':<18} : {int(tables_df['Merged Cells'].sum())}")
    print(f"{'Mean fill rate':<18} : "
          f"{_structured['Fill Rate'].mean():.1%} (0x0 제외 {len(_structured)}건 기준)")

print("\n[문서별 요약]")
display(summary_df)

DOCLING PARSING TEST SUMMARY
Documents          : 11 / 11 변환 성공
Pages              : 222
Elapsed            : 23.2s

[1] Layout Analysis
----------------------------------------
List               : 2432
Text               : 738
Heading            : 493
Footer             : 431
Header             : 214
Table              : 54
Code               : 14
Picture            : 7
Formula            : 3
Caption            : 2

[2] Text Hierarchy
----------------------------------------
Heading detected   : 493
H1                 : 138
H2                 : 355
Level jumps        : 0
Flat documents     : 3 / 11

[3] Table Structure
----------------------------------------
Tables detected    : 54
Tables converted   : 54
Structure OK       : 47 / 54   (0x0 = 영역만 검출: 7)
Rows reconstructed : 224
Columns (max)      : 7
Header detected    : 45 / 54
Multi header       : 0
Merged cells       : 8
Mean fill rate     : 100.0% (0x0 제외 47건 기준)

[문서별 요약]


,Document,Folder,Pages,Status,Sec,Elements,Title,Heading,Text,List,...,Picture,Header/Footer,Max Level,Level Jumps,Flat?,Tables Converted,Tables 0x0,Table Rows,Merged Cells,Header Detected
0,법인카드_사용규정,pdf,7,SUCCESS,3.8,120,0,28,11,60,...,0,18,2,0,no,2,0,9,0,1
1,부서소개,pdf,5,SUCCESS,1.2,32,0,9,1,0,...,0,13,1,0,YES (계층 없음),9,0,36,2,9
2,업무추진비_사용규정,pdf,6,SUCCESS,0.5,101,0,25,18,41,...,0,15,2,0,no,2,0,9,0,2
3,조직도,pdf,6,SUCCESS,0.6,42,0,9,3,9,...,1,15,1,0,YES (계층 없음),4,0,18,0,4
4,조직설계_상세기획서,pdf,12,SUCCESS,2.0,101,0,23,7,25,...,1,33,2,0,no,12,0,68,3,12
5,직급체계,pdf,4,SUCCESS,0.6,31,0,6,3,9,...,0,10,1,0,YES (계층 없음),3,0,17,1,3
6,출장비_사용규정,pdf,6,SUCCESS,0.6,83,0,24,12,28,...,0,15,2,0,no,4,0,11,0,3
7,회식_운영규정,pdf,10,SUCCESS,2.0,105,0,21,24,21,...,0,28,2,0,no,11,0,56,2,11
8,법인세법,law,97,SUCCESS,7.6,2098,0,188,387,1216,...,3,291,2,0,no,4,4,0,0,0
9,부가가치세법,law,35,SUCCESS,1.9,767,0,57,105,493,...,1,105,2,0,no,3,3,0,0,0


---
## 8. Markdown 결과 저장

문서별 Markdown은 `output/markdown/<문서>.md`, 전체를 이어붙인 것은 `output/docling_result.md`에
저장한다. 저장 후 미리보기 문서의 앞부분을 화면에 찍어 계층·표가 Markdown에도 살아 있는지 눈으로 확인한다.

In [11]:
MD_PREVIEW_CHARS = 1200

md_parts, md_rows = [], []
for name, doc in DOCS.items():
    try:
        markdown = doc.export_to_markdown()
    except Exception as exc:  # noqa: BLE001
        print(f"[error] Markdown export 실패: {name} — {type(exc).__name__}: {exc}")
        continue
    (MD_DIR / f"{name}.md").write_text(markdown, encoding="utf-8")
    md_parts.append(f"# {name}\n\n{markdown}")
    md_rows.append({"Document": name, "Chars": len(markdown), "Lines": markdown.count(chr(10)) + 1})

combined_md = "\n\n---\n\n".join(md_parts)
(OUTPUT_DIR / "docling_result.md").write_text(combined_md, encoding="utf-8")

print(f"문서별 Markdown : {MD_DIR.relative_to(BASE_DIR)}/<문서>.md  ({len(md_rows)}건)")
print(f"통합 Markdown   : {(OUTPUT_DIR / 'docling_result.md').relative_to(BASE_DIR)}  "
      f"({len(combined_md):,} chars)")
display(pd.DataFrame(md_rows))

print("\n" + "=" * 78)
print(f"MARKDOWN PREVIEW — {PREVIEW_DOC} (앞 {MD_PREVIEW_CHARS}자)")
print("=" * 78)
print((MD_DIR / f"{PREVIEW_DOC}.md").read_text(encoding="utf-8")[:MD_PREVIEW_CHARS])

문서별 Markdown : output\markdown/<문서>.md  (11건)
통합 Markdown   : output\docling_result.md  (372,301 chars)


,Document,Chars,Lines
0,법인카드_사용규정,6288,168
1,부서소개,5723,73
2,업무추진비_사용규정,6849,148
3,조직도,3754,66
4,조직설계_상세기획서,15446,191
5,직급체계,2859,53
6,출장비_사용규정,5420,128
7,회식_운영규정,15934,192
8,법인세법,188116,2513
9,부가가치세법,64684,889



MARKDOWN PREVIEW — 법인카드_사용규정 (앞 1200자)
## 타 이 거 주 식 회 사 ( T i ge r I n c . )

## 법인카드 사용 규정

| 제정일    | 2026. 7. 20.   |
|-----------|----------------|
| 시행일    | 2026. 8. 1.    |
| 소관부서  | 경영지원본부   |

개정이력 v1.0 제정 (조문 정합성 검토 반영) / v1.1 개정: 제2조 관리자 정의 정비, 제10조 사전승인 예외절차 신 설, 별표1 이사 겸직·대행 시 한도 적용기준 명확화

※ 본 규정은 데모/교육 목적의 가상 기업(타이거 주식회사, Tiger Inc.) 시나리오용으로 작성되었으며, 회사는 설립 12년차 중견 기업(세법상 중소기업 특례 대상 아님)임을 가정합니다. 연 매출액은 연결 기준 약 1,200억원(500억원 초과 구간)으로 가정하였 으며, 실제 적용 시 회사의 정확한 매출 규모·업종·지주회사 여부 등에 따라 한도 및 조항을 조정해야 합니다.

### 제1조 (목적)

이 규정은 타이거 주식회사(이하 "회사")가 임직원에게 발급하는 법인카드의 발급, 관리, 사용 및 정산에 관 한 사항을 정함으로써 업무 효율성을 높이고, 관계 법령에 따른 손금 인정 요건을 충족하여 세무상 불이익을 방지하며, 법인카드 오남용을 예방하는 것을 목적으로 한다.

### 제2조 (정의) 개정 v1.1

이 규정에서 사용하는 용어의 정의는 다음과 같다.

1. "법인카드"란 회사 명의로 발급되어 임직원이 업무 목적의 지출에 사용하는 신용카드·체크카드를 말한 다.
2. "사용자"란 법인카드를 발급받아 사용하는 임직원을 말한다.
3. "관리자"란 법인카드 정산 승인 권한을 가진 자로서 「직급체계」에 따라 결재권을 보유한 팀장·부서장· 본부장 및 대표이사를 말한다. 법인카드의 발급·회수 등 관리업무는 경영지원본부(인사부·재무회계부·총무 구매부·법무부)가 총괄한다.
4. "담당자"란 법인카드를 발급받아 사용하고 정산을 신청

### 8-1. 생성된 산출물 목록

노트북이 만든 파일은 전부 `docling_eval/output/` 아래에만 있다.

In [12]:
artifacts = sorted(
    (p for p in OUTPUT_DIR.rglob("*") if p.is_file()),
    key=lambda p: p.relative_to(OUTPUT_DIR).as_posix(),
)
artifact_df = pd.DataFrame(
    [
        {"File": p.relative_to(OUTPUT_DIR).as_posix(), "KB": round(p.stat().st_size / 1024, 1)}
        for p in artifacts
    ]
)
print(f"총 {len(artifact_df)}개 파일, {artifact_df['KB'].sum():,.1f} KB  ({OUTPUT_DIR})")
display(artifact_df.head(40))

총 137개 파일, 4,276.1 KB  (D:\project\SKN29-FINAL-1TEAM\docling_eval\output)


,File,KB
0,docling_result.md,793.2
1,hierarchy/hierarchy_result.csv,852.2
2,hierarchy/법인세법_tree.txt,247.8
3,hierarchy/법인카드_사용규정_tree.txt,11.4
4,hierarchy/부가가치세법_tree.txt,95.5
5,hierarchy/부서소개_tree.txt,0.7
6,hierarchy/업무추진비_사용규정_tree.txt,9.9
7,hierarchy/여신전문금융업법_tree.txt,103.4
8,hierarchy/조직도_tree.txt,2.4
9,hierarchy/조직설계_상세기획서_tree.txt,6.2


---
## 최종 판정 — 세 질문에 답하기

```text
┌──────────────────────────────────────┐
│ 1. Layout Analysis                   │
│    → 영역 구분이 정확한가?           │
├──────────────────────────────────────┤
│ 2. Text Hierarchy                    │
│    → 문서 계층이 유지되는가?         │
├──────────────────────────────────────┤
│ 3. Table Structure                   │
│    → 표 구조가 복원되는가?           │
└──────────────────────────────────────┘
```

| 질문 | 어디를 보나 | 나쁨 신호 |
|---|---|---|
| ① 영역 구분 | §4 `layout_by_document` / `layout_totals`, `layout/layout_result.csv` | `Other` 비중이 크다 · Heading이 0인 문서 · 본문이 전부 `Text` 한 덩어리 |
| ② 계층 보존 | §5 `heading_level_df` / `hierarchy_metrics_df`, `hierarchy/<문서>_tree.txt` | `Flat? = YES` · `Level Jumps`가 헤딩 수에 맞먹음 · 트리에서 본문이 엉뚱한 헤딩에 붙음 |
| ③ 표 복원 | §6 `tables_df`, `tables/<문서>/table_NN.json` | `Header = Not detected` 비중이 크다 · `Fill Rate`가 낮다 · Rows/Cols가 원본과 다르다 |

**주의 — 이 노트북이 재는 것과 재지 않는 것.** 여기서 나오는 수치는 전부 Docling의 *자기 보고*다.
원본 PDF와의 대조(정답지 채점)는 하지 않는다. 최종 채택 판단 전에는 `tables/*.csv`와
`hierarchy/*_tree.txt` 몇 건을 원본 PDF와 눈으로 대조하는 단계가 반드시 필요하다.

또한 이 규정 PDF들에는 docling 리딩오더·CJK 줄바꿈·목록 마커 관련 결함이 알려져 있다
(CLAUDE.md 상태 보드 "RAG 문서 파싱(docling)" 행 참조). 그 교정은 이 평가 노트북의 범위 밖이다 —
여기서는 **후처리 없는 날것의 Docling 성능**을 본다.